# Optional: under the hood (ARC-Challenge, no framework)

**Companion to `01_inspect_truthfulqa.ipynb`.** Inspect hides the mechanics. This notebook does the same kind of eval by hand in about 40 lines, so you can see what happens inside.

It runs **Qwen2.5-0.5B-Instruct** on **ARC-Challenge** (1,172 multiple-choice science questions). Instead of letting the model write `ANSWER: B` and parsing it, as Inspect does, it reads the model's **probabilities** for A, B, C and D directly. That unlocks one diagnostic Inspect's text scoring can't give you: **calibration**.

**Why calibration matters for AI Control.** The group's project is a *small, trusted model* overseeing a *stronger, untrusted* one. This model is the kind that would be the trusted monitor. A monitor has to **know when it's unsure** so it can escalate. Diagnostics here:

1. **Accuracy** with an error bar, against chance
2. **Format check**: is it actually answering with a letter?
3. **Position bias**: does it over-pick one letter?
4. **Calibration**: when it's 80% confident, is it right 80% of the time?
5. **Confidently wrong**: what a weak monitor would wave through
6. *(Optional)* **Rotation robustness**: does the answer survive moving the options?

**Before you start:** `Runtime → Change runtime type → T4 GPU`. Then step through cell by cell.

## 0. Setup

Colab already has PyTorch, `transformers` and `datasets`. This line just makes sure they're present. It's quick if they are.

In [ ]:
!pip -q install transformers datasets accelerate

## 1. Settings

The only cell you need to edit. Start with the defaults. To compare a bigger model later, change `MODEL_ID` to `"Qwen/Qwen2.5-1.5B-Instruct"` and run everything again: each run saves its own results files.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # any Hugging Face causal LM works
N_QUESTIONS = None   # None = all 1,172 questions. Use e.g. 100 for a fast test run.
SEED = 0             # makes the random parts (sampling, bootstrap) repeatable

In [ ]:
import json, random, time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32   # half precision on GPU halves memory
print("device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "no GPU (this will be slow)")

## 2. Load the benchmark

ARC ("AI2 Reasoning Challenge") comes from the Allen Institute for AI. The **Challenge** split holds the questions that simple retrieval and word-matching baselines got wrong. Look at the raw format first. Never trust a dataset you haven't looked at.

In [ ]:
ds = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")
print(len(ds), "questions")
ds[0]

Each row has a `question`, a list of `choices` (texts plus labels) and an `answerKey`. Two quirks you'd only catch by looking:

- A few questions use labels `1, 2, 3, 4` instead of `A, B, C, D`.
- A few have 3 or 5 options instead of 4.

So we convert every row to one clean shape: the option texts, plus the **position** (0, 1, 2…) of the right answer. We then re-letter everything ourselves as A, B, C…

In [ ]:
LETTERS = "ABCDE"

def to_item(row):
    labels, texts = row["choices"]["label"], row["choices"]["text"]
    return {"id": row["id"],
            "question": row["question"],
            "options": texts,
            "answer_idx": labels.index(row["answerKey"])}   # position of the correct option

items = [to_item(r) for r in ds]
if N_QUESTIONS:
    items = random.Random(SEED).sample(items, N_QUESTIONS)   # random subset, not just the first N

print(len(items), "questions to run")
print("options per question:", dict(Counter(len(it["options"]) for it in items)))
print("correct-answer positions:", dict(sorted(Counter(LETTERS[it["answer_idx"]] for it in items).items())))

## 3. Turn a question into a prompt

We show the question and lettered options, then end with `Answer:`. The model's **very next token** is its answer.

We use a plain-text prompt instead of the model's chat template on purpose. It keeps exactly one thing between you and the model: this string, which you can read.

In [ ]:
def make_prompt(item):
    lines = ["The following is a multiple choice science question. Answer with the letter of the correct option.",
             "",
             f"Question: {item['question']}"]
    for letter, text in zip(LETTERS, item["options"]):
        lines.append(f"{letter}. {text}")
    lines.append("Answer:")
    return "\n".join(lines)

print(make_prompt(items[0]))

## 4. Load the model

Downloads about 1 GB the first time. `.eval()` switches off training-only behaviour such as dropout.

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=dtype).to(device).eval()
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters")

## 5. How we read the model's answer

A language model outputs a **logit** (a raw score) for every token in its vocabulary, about 150,000 of them for Qwen. We don't let it generate freely. We look only at the scores for the tokens `" A"`, `" B"`, `" C"`, `" D"` (with the leading space, because they follow `Answer:`), then:

- **prediction** = the letter with the highest score
- **confidence** = softmax over just those letters, which turns the scores into probabilities that sum to 1
- **letter mass** = how much of the model's *total* next-token probability landed on any letter. If this is low, the model wanted to say something else and isn't following the format.

This is called **log-likelihood scoring**. It's deterministic, fast, and never needs answer parsing. First, check that each letter is exactly one token:

In [ ]:
letter_ids = [tok.encode(" " + L, add_special_tokens=False) for L in LETTERS]
assert all(len(t) == 1 for t in letter_ids), letter_ids   # if this fails, the scoring idea breaks
letter_ids = [t[0] for t in letter_ids]
print(dict(zip(LETTERS, letter_ids)))

Sanity check before running 1,172 questions: what does the model *want* to say next on one example?

In [ ]:
with torch.no_grad():
    inputs = tok(make_prompt(items[0]), return_tensors="pt").to(device)
    next_logits = model(**inputs).logits[0, -1]          # scores for the token after "Answer:"
    top = torch.softmax(next_logits.float(), dim=0).topk(8)

for p, i in zip(top.values, top.indices):
    print(f"{tok.decode(int(i))!r:>12}  {p.item():.3f}")
print("correct answer:", LETTERS[items[0]["answer_idx"]])

## 6. Run the benchmark

This is the whole eval. One forward pass per question; no batching, to keep it readable. About 1 minute on a T4 for the 0.5B model.

In [ ]:
@torch.no_grad()
def score(item):
    inputs = tok(make_prompt(item), return_tensors="pt").to(device)
    logits = model(**inputs).logits[0, -1].float()
    ids = letter_ids[:len(item["options"])]               # only the letters this question uses
    probs = torch.softmax(logits[ids], dim=0)             # confidence among the options
    letter_mass = torch.softmax(logits, dim=0)[ids].sum() # share of ALL probability on valid letters
    return probs.cpu().numpy(), letter_mass.item()

rows, t0 = [], time.time()
for i, item in enumerate(items):
    probs, mass = score(item)
    pred = int(probs.argmax())
    rows.append({"id": item["id"],
                 "n_options": len(item["options"]),
                 "answer": LETTERS[item["answer_idx"]],
                 "pred": LETTERS[pred],
                 "correct": pred == item["answer_idx"],
                 "confidence": float(probs.max()),
                 "p_correct": float(probs[item["answer_idx"]]),
                 "letter_mass": mass})
    if (i + 1) % 200 == 0:
        print(f"{i + 1}/{len(items)}  running accuracy {np.mean([r['correct'] for r in rows]):.3f}")

df = pd.DataFrame(rows)
print(f"done in {time.time() - t0:.0f}s")
df.head()

## 7. Diagnostic 1: accuracy, with an error bar

A score without an error bar can't tell you whether two models really differ. We **bootstrap**: resample the questions with replacement 2,000 times, recompute accuracy each time, and take the middle 95%.

**Chance** is the average of 1/(number of options). That's the score of a model that guesses.

In [ ]:
rng = np.random.default_rng(SEED)
c = df["correct"].to_numpy()
boot = rng.choice(c, size=(2000, len(c)), replace=True).mean(axis=1)
acc, lo, hi = c.mean(), *np.percentile(boot, [2.5, 97.5])
chance = (1 / df["n_options"]).mean()

print(f"accuracy  {acc:.3f}   95% CI [{lo:.3f}, {hi:.3f}]   (n = {len(c)})")
print(f"chance    {chance:.3f}")
print(f"mean letter mass {df['letter_mass'].mean():.3f}   (share of next-token probability on a valid letter)")

**How to read it.** Is the CI entirely above chance? How wide is it? Width shrinks with the square root of *n*, which is why a 100-question run gives a much blurrier answer. If letter mass is well below 1, part of the model's "wrong" answers may be format failures, not knowledge failures. Note that before blaming the model.

## 8. Diagnostic 2: position bias

If the model knew nothing and always said "A", it would still score whatever share of answers happen to be "A". Compare **how often it predicts each letter** with **how often each letter is correct**. We use 4-option questions only, so the letters are comparable.

In [ ]:
four = df[df["n_options"] == 4]
bias = pd.DataFrame({"predicted": four["pred"].value_counts(normalize=True),
                     "actually correct": four["answer"].value_counts(normalize=True)}).reindex(list("ABCD")).fillna(0)
print(bias.round(3))

ax = bias.plot.bar(rot=0, figsize=(6, 3.5), title=f"Position bias: {MODEL_ID.split('/')[-1]}")
ax.set_ylabel("share of questions"); plt.tight_layout(); plt.show()

## 9. Diagnostic 3: calibration

Group the answers by confidence (for example, every answer given with 60–70% confidence) and check how often each group was right. A perfectly **calibrated** model lands on the diagonal. **ECE** (expected calibration error) is the average gap, weighted by how many answers fall in each bin.

This matters most for the AI Control project. A trusted monitor needs to *know when it doesn't know*, so it can escalate instead of waving something through.

In [ ]:
bins = np.linspace(0, 1, 11)
df["bin"] = pd.cut(df["confidence"], bins, include_lowest=True)
cal = df.groupby("bin", observed=True).agg(n=("correct", "size"), confidence=("confidence", "mean"), accuracy=("correct", "mean"))
ece = (cal["n"] / cal["n"].sum() * (cal["confidence"] - cal["accuracy"]).abs()).sum()
print(cal.round(3))
print(f"\nECE = {ece:.3f}   (0 = perfectly calibrated)")

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot([0, 1], [0, 1], "--", color="gray", label="perfect calibration")
ax.plot(cal["confidence"], cal["accuracy"], "o-", label=MODEL_ID.split("/")[-1])
ax.set_xlabel("average confidence in bin"); ax.set_ylabel("accuracy in bin")
ax.set_title("Reliability diagram"); ax.legend(); plt.tight_layout(); plt.show()

Points **below** the diagonal mean overconfidence: it says 80%, but it's right less often. Bins with only a handful of answers are noise, so check the `n` column before reading anything into a point.

## 10. Diagnostic 4: confidently wrong

The most informative rows for oversight work: questions where the model was **sure and wrong**. Read them. Is there a pattern (negation, units, multi-step reasoning, a trap option)?

In [ ]:
by_id = {it["id"]: it for it in items}
worst = df[~df["correct"]].sort_values("confidence", ascending=False).head(5)

for _, r in worst.iterrows():
    it = by_id[r["id"]]
    print(f"[{r['confidence']:.0%} confident]  {it['question']}")
    for L, text in zip(LETTERS, it["options"]):
        mark = "  <- model" if L == r["pred"] else ("  <- correct" if L == r["answer"] else "")
        print(f"    {L}. {text}{mark}")
    print()

## 11. Diagnostic 5 (optional): does the answer survive shuffling?

Rotate every question's options by one position (A→B, B→C, …, last→A) and re-score. A model that actually knows the answer should pick the **same option text** both times. **Agreement** is the share of questions where it did. It's about 2× the original run time, so we use the first 200 questions.

In [ ]:
def rotate(item):
    k = len(item["options"])
    return {**item,
            "options": [item["options"][(j - 1) % k] for j in range(k)],   # last option moves to the front
            "answer_idx": (item["answer_idx"] + 1) % k}

subset = items[:200]
agree, correct_rot = [], []
for item in subset:
    p0, _ = score(item)
    p1, _ = score(rotate(item))
    k = len(item["options"])
    agree.append((int(p0.argmax()) + 1) % k == int(p1.argmax()))   # same option text chosen?
    correct_rot.append(int(p1.argmax()) == (item["answer_idx"] + 1) % k)

print(f"agreement after rotation: {np.mean(agree):.3f}")
print(f"accuracy on rotated set:  {np.mean(correct_rot):.3f}   vs original on same questions: {df['correct'][:len(subset)].mean():.3f}")

## 12. Save and summarize

Saves per-question results (CSV) and a summary (JSON), named after the model, and downloads both. The printout at the end is what to bring Thursday.

In [ ]:
tag = MODEL_ID.split("/")[-1]
summary = {"model": MODEL_ID, "benchmark": "ARC-Challenge (test)", "n": int(len(df)),
           "accuracy": round(float(acc), 4), "ci95": [round(float(lo), 4), round(float(hi), 4)],
           "chance": round(float(chance), 4), "ece": round(float(ece), 4),
           "mean_letter_mass": round(float(df["letter_mass"].mean()), 4),
           "pred_share_ABCD": bias["predicted"].round(3).to_dict(),
           "scoring": "next-token log-likelihood over option letters, zero-shot"}
if "agree" in globals():
    summary["rotation_agreement_200"] = round(float(np.mean(agree)), 4)

df.drop(columns="bin").to_csv(f"results_{tag}.csv", index=False)
with open(f"summary_{tag}.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
try:
    from google.colab import files
    files.download(f"results_{tag}.csv"); files.download(f"summary_{tag}.json")
except ImportError:
    print("(not on Colab: files saved next to the notebook)")

## 13. (Optional) Compare models

Change `MODEL_ID` in section 1 to `Qwen/Qwen2.5-1.5B-Instruct`, then `Runtime → Run all` again. This cell lines up every `summary_*.json` in the session. It's a two-point preview of the capability-gap question the project cares about: how much does the overseer improve with size?

In [ ]:
import glob
table = pd.DataFrame([json.load(open(p)) for p in sorted(glob.glob("summary_*.json"))])
table[[c for c in ["model", "n", "accuracy", "ci95", "chance", "ece", "mean_letter_mass", "rotation_agreement_200"] if c in table]]